# 🎙️ ¿Cómo transformar fuentes sonoras en textos explotables para la investigación?

## Guía metodológica para la transcripción automática de audio con Whisper *(OpenAI)*

**Echavarría, Andrés.**

**Grupo de interés en adquisición asistida de textos** · Consorcio Huma-Num **ARIANE**

---

### Planteamiento del problema

Gran parte de la producción intelectual, patrimonial y testimonial de las ciencias humanas y sociales existe **únicamente en forma oral**: entrevistas de campo, ponencias, conferencias grabadas, emisiones de radio, testimonios audiovisuales, fondos sonoros de archivos y bibliotecas. Estos materiales —a menudo irrepetibles— constituyen fuentes primarias de enorme valor, pero permanecen en buena medida **inaccesibles a la búsqueda, la citación y el análisis textual** mientras no se transcriban.

La transcripción manual es lenta, costosa y difícilmente escalable. Surge entonces una **pregunta de operativa**:

> *¿En qué medida las herramientas actuales de reconocimiento automático del habla (ASR) permiten obtener transcripciones fiables de fuentes orales en español, francés y otras lenguas de trabajo de nuestras comunidades de investigación, y cuáles son los parámetros, límites y buenas prácticas que debemos conocer para integrarlas en nuestros flujos de trabajo?*
---
### Qué propone este cuaderno Jupyter

Este cuaderno de Jupyter para Google Colab ofrece una **guía práctica y razonada** para utilizar [**Whisper**](https://github.com/openai/whisper), el modelo de reconocimiento automático del habla desarrollado por OpenAI y publicado en código abierto (licencia MIT). Whisper fue entrenado con más de **680 000 horas** de audio multilingüe etiquetado, lo que le confiere tres capacidades fundamentales para la investigación:

| Capacidad | Descripción | Utilidad para la investigación |
|-----------|-------------|-------------------------------|
| **Transcripción** | Convierte audio hablado en texto en el idioma original | Producir versiones textuales de entrevistas, conferencias, testimonios orales |
| **Traducción** | Traduce el audio de cualquier lengua al inglés | Facilitar el acceso preliminar a fuentes en lenguas que el investigador no domina |
| **Detección de idioma** | Identifica automáticamente la lengua hablada | Clasificar fondos sonoros multilingües sin escucha previa |
---
### Estructura de la guía

El cuaderno se organiza en **nueve secciones** que siguen el recorrido completo de un flujo de trabajo de transcripción:

1. **Configuración del entorno** — Activar la GPU y verificar los recursos de cómputo disponibles.
2. **Instalación de dependencias** — Instalar Whisper, FFmpeg y yt-dlp.
3. **Modelos disponibles** — Comparar los seis tamaños de modelo (de `tiny` a `large` y `turbo`) según la relación calidad/velocidad/memoria.
4. **Montar Google Drive** — Conectar el espacio de almacenamiento personal para leer y guardar archivos.
5. **Obtener el audio** — Tres vías: subir un archivo local, usar un archivo de Drive o descargar audio de YouTube.
6. **Verificar y escuchar el audio** — Controlar la duración, el formato y la calidad antes de lanzar el proceso.
7. **Transcribir con Whisper** — Dos métodos (línea de comandos y script Python), con ejemplos para transcripción básica, transcripción rápida, vocabulario especializado, traducción al inglés y gestión de audios problemáticos.
8. **Explorar y descargar los resultados** — Formatos de salida (`.txt`, `.srt`, `.vtt`, `.tsv`, `.json`) y cómo leer o exportar cada uno.
9. **Consejos y resolución de problemas** — Buenas prácticas ante ruido, alucinaciones, audios largos o errores de memoria.
---
### Condiciones de uso

- **Entorno recomendado**: Google Colab con GPU (T4 o superior).
- **Costo**: El uso de Whisper es gratuito (modelo abierto). Solo se consume la cuota de GPU de Colab.
- **Formatos de entrada**: `.mp3`, `.wav`, `.flac`, `.ogg`, `.m4a`, `.wma`, `.aac`, y archivos de video (`.mp4`, `.mkv`, `.avi`, etc.).
- **Lenguas principales soportadas**: español, francés, inglés, portugués, alemán, italiano, catalán, gallego, euskera, entre [muchas otras](https://github.com/openai/whisper#available-models-and-languages).

### **¿Por qué usar Google Colab?**

Los modelos más grandes de Whisper (como `large` o `large-v3`) necesitan alrededor de **10 GB de VRAM** (memoria de la tarjeta gráfica). La mayoría de computadores personales no tienen GPU suficiente, pero Colab nos "~~presta~~" una **GPU NVIDIA T4 con 16 GB de VRAM** de forma "~~gratuita~~".

> ⚠️ **Nota sobre el signo `!`:** En los cuadernos de Colab, las líneas que comienzan con `!` se ejecutan como **comandos de terminal** (shell), no como código Python. Es el equivalente a escribir en una consola/terminal.

---
---
### Cómo citar esta guía

> Echavarría, Andrés. *Guía metodológica para la transcripción automática de audio con Whisper (OpenAI)* Grupo de interés en adquisición automática de textos, Consorcio Huma-Num ARIANE. [Cuaderno Jupyter Google Colab], versión 1.0, marzo de 2026.
---
---


> 💡 **Nota**: Esta guía es un *documento vivo*. Les invitamos a reportar errores, sugerir mejoras o proponer nuevos ejemplos de uso a través del grupo de trabajo.


---
# ⚙️ 1. Configuración del entorno de ejecución

Antes de ejecutar cualquier celda, hay que activar la **GPU** en Colab:

1. En el menú superior, ir a **Entorno de ejecución → Cambiar tipo de entorno de ejecución**.
2. En la ventana emergente, seleccionar **GPU** como acelerador de hardware (normalmente una **T4**).
3. Hacer clic en **Guardar**.

Después de guardar, el entorno se reiniciará. Esto es normal.

### Verificar la GPU asignada
Ejecuta la siguiente celda para confirmar que tienes una GPU disponible:

In [ ]:
# Verificar la GPU asignada por Colab
!nvidia-smi

# También podemos verificarlo con PyTorch
import torch
print("\n" + "="*50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU disponible: {gpu_name}")
    print(f"   Memoria total: {gpu_mem:.1f} GB")
else:
    print("❌ No se detectó GPU. Revisa la configuración del entorno.")
print("="*50)

---
# 📦 2. Instalación de dependencias

Necesitamos instalar:
- **openai-whisper**: el modelo de transcripción.
- **ffmpeg**: herramienta para procesar archivos de audio/video (suele venir preinstalado en Colab, pero lo aseguramos).
- **yt-dlp**: para descargar audio de YouTube u otras plataformas de video.

> 💡 La opción `-qqq` silencia la salida de la instalación para no llenar la pantalla de mensajes.

In [ ]:
# 1) Asegurar que ffmpeg está instalado
!apt-get update -qq && apt-get install -y -qq ffmpeg

# 2) Instalar Whisper desde el repositorio oficial de OpenAI
#    Esto garantiza tener la última versión disponible
!pip install -U openai-whisper -qqq

# 3) Instalar yt-dlp para poder descargar audio de YouTube
!pip install -U yt-dlp -qqq

print("\n✅ Todas las dependencias instaladas correctamente.")

---
# 🧠 3. Modelos disponibles en Whisper

Whisper ofrece **seis tamaños de modelo** (más variantes). A mayor tamaño, mejor calidad de transcripción, pero más lento y más consumo de VRAM.

| Modelo | Parámetros | VRAM necesaria | Velocidad relativa | Multilingüe | Solo inglés |
|--------|-----------|----------------|-------------------|-------------|-------------|
| `tiny` | 39 M | ~1 GB | ~10× | ✅ `tiny` | ✅ `tiny.en` |
| `base` | 74 M | ~1 GB | ~7× | ✅ `base` | ✅ `base.en` |
| `small` | 244 M | ~2 GB | ~4× | ✅ `small` | ✅ `small.en` |
| `medium` | 769 M | ~5 GB | ~2× | ✅ `medium` | ✅ `medium.en` |
| `large` | 1 550 M | ~10 GB | 1× | ✅ `large` | ❌ |
| **`turbo`** | **809 M** | **~6 GB** | **~8×** | ✅ `turbo` | ❌ |

### ¿Cuál elegir?

- **Para transcripción rápida** (borradores, notas): `small` o `turbo`.
- **Para la mejor calidad posible**: `large` (equivalente a `large-v3`, la última versión).
- **Para traducción al inglés**: usar `medium` o `large` (⚠️ `turbo` **no** está entrenado para traducir).
- **Solo inglés**: los modelos `.en` (ej. `small.en`) son ligeramente mejores para inglés puro.
- **En la GPU T4 gratuita de Colab (16 GB VRAM)**: todos los modelos caben sin problema.

> 💡 **Recomendación LA RANA**: para audios complejos o en varios idiomas, usar **`large`**. Para audios largos pero simples, **`turbo`** es un buen compromiso entre velocidad y calidad.

---
# 📂 4. Montar Google Drive (opcional)

Si tus archivos de audio están en Google Drive, o si quieres guardar ahí los resultados, ejecuta la siguiente celda. Colab te pedirá autorización para acceder a tu Drive.

Una vez montado, tus archivos estarán accesibles en la ruta:
```
/content/drive/MyDrive/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verificar que se montó correctamente
import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive montado en /content/drive/MyDrive/")
else:
    print("❌ No se pudo montar Google Drive.")

---
# 📤 5. Obtener el audio a transcribir

Hay **tres formas** de obtener el audio:

### Opción A: Subir un archivo desde tu computador
### Opción B: Usar un archivo que ya está en Google Drive
### Opción C: Descargar el audio de un video de YouTube

> **Formatos de audio soportados por Whisper**: `.mp3`, `.wav`, `.flac`, `.ogg`, `.m4a`, `.wma`, `.aac` y prácticamente cualquier formato que `ffmpeg` pueda leer (incluyendo archivos de video `.mp4`, `.mkv`, `.avi`, etc., de los cuales extraerá automáticamente la pista de audio).

### 📤 Opción A: Subir un archivo desde tu computador
Ejecuta la celda siguiente. Aparecerá un botón **"Seleccionar archivos"** para subir tu audio.

In [ ]:
from google.colab import files

# Subir archivo(s) de audio
uploaded = files.upload()

# Mostrar los archivos subidos
for filename in uploaded.keys():
    print(f"✅ Archivo subido: {filename} ({len(uploaded[filename])/1e6:.1f} MB)")

# El archivo queda en /content/nombre_del_archivo
AUDIO_FILE = list(uploaded.keys())[0]  # Tomamos el primero
print(f"\n🎯 Archivo seleccionado para transcribir: {AUDIO_FILE}")

### 📁 Opción B: Usar un archivo de Google Drive
Si tu audio está en Drive, simplemente indica la ruta completa. Por ejemplo:
```python
AUDIO_FILE = "/content/drive/MyDrive/mis_audios/entrevista.mp3"
```

In [ ]:
# ✏️ EDITA AQUÍ la ruta a tu archivo en Google Drive
# Ejemplo: AUDIO_FILE = "/content/drive/MyDrive/carpeta/mi_audio.mp3"

AUDIO_FILE = "/content/drive/MyDrive/mi_audio.mp3"  # ← Cambia esta ruta

import os
if os.path.exists(AUDIO_FILE):
    size_mb = os.path.getsize(AUDIO_FILE) / 1e6
    print(f"✅ Archivo encontrado: {AUDIO_FILE} ({size_mb:.1f} MB)")
else:
    print(f"❌ Archivo no encontrado: {AUDIO_FILE}")
    print("   Verifica la ruta. Puedes explorar tu Drive en el panel izquierdo (icono de carpeta).")

### 🎬 Opción C: Descargar audio de un video de YouTube

Usamos **yt-dlp**, la herramienta más robusta para descargar contenido de YouTube y otras plataformas. Extraeremos **solo la pista de audio** en formato MP3 (más ligero que descargar todo el video).

> ⚠️ **Aviso legal**: descargar contenido de YouTube puede estar sujeto a restricciones de derechos de autor. Usa esta función solo con contenido del que tengas derechos o permiso.

In [ ]:
# ✏️ EDITA AQUÍ: pega la URL del video de YouTube
YOUTUBE_URL = "https://www.youtube.com/watch?v=XXXXXXXXXX"  # ← Cambia esta URL

# Nombre del archivo de salida (sin extensión)
OUTPUT_NAME = "audio_youtube"

import subprocess

# Descargar solo el audio, convertirlo a MP3 de alta calidad
cmd = [
    "yt-dlp",
    "--extract-audio",           # Extraer solo audio
    "--audio-format", "mp3",     # Convertir a MP3
    "--audio-quality", "0",      # Máxima calidad (VBR ~245 kbps)
    "-o", f"/content/{OUTPUT_NAME}.%(ext)s",  # Ruta de salida
    YOUTUBE_URL
]

print("⏳ Descargando audio del video...")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    AUDIO_FILE = f"/content/{OUTPUT_NAME}.mp3"
    print(f"✅ Audio descargado: {AUDIO_FILE}")
    import os
    if os.path.exists(AUDIO_FILE):
        size_mb = os.path.getsize(AUDIO_FILE) / 1e6
        print(f"   Tamaño: {size_mb:.1f} MB")
else:
    print("❌ Error al descargar:")
    print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)

---
# 🔊 6. Verificar y escuchar el audio

Antes de lanzar la transcripción, es buena idea verificar que el archivo se cargó bien y escucharlo brevemente:

In [ ]:
import subprocess, json
from IPython.display import Audio, display

# Obtener información del archivo con ffprobe
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", "-show_streams", AUDIO_FILE],
    capture_output=True, text=True
)

if probe.returncode == 0:
    info = json.loads(probe.stdout)
    fmt = info.get("format", {})
    duration_sec = float(fmt.get("duration", 0))
    duration_min = duration_sec / 60
    print(f"📄 Archivo: {AUDIO_FILE}")
    print(f"⏱️  Duración: {int(duration_min)}m {int(duration_sec % 60)}s ({duration_sec:.1f} segundos)")
    print(f"📦 Tamaño: {float(fmt.get('size', 0))/1e6:.1f} MB")
    print(f"🎵 Formato: {fmt.get('format_long_name', 'desconocido')}")
    for stream in info.get("streams", []):
        if stream.get("codec_type") == "audio":
            print(f"🔈 Codec: {stream.get('codec_name', '?')} | "
                  f"Sample rate: {stream.get('sample_rate', '?')} Hz | "
                  f"Canales: {stream.get('channels', '?')}")
    print()

# Reproducir el audio directamente en el cuaderno
print("▶️  Reproducir audio (primeros 30 segundos):")
display(Audio(AUDIO_FILE))

---
# ✍️ 7. Transcribir con Whisper

Hay **dos formas** de usar Whisper:
- **Método A** — Línea de comandos (`!whisper ...`): más sencillo, ideal para transcripciones rápidas.
- **Método B** — Script Python (`import whisper`): más flexible, permite manipular el resultado en código.

---

## Método A: Línea de comandos (recomendado para empezar)

### Opciones principales

| Opción | Descripción | Valores posibles |
|--------|-------------|------------------|
| `--model` | Tamaño del modelo | `tiny`, `base`, `small`, `medium`, `large`, `turbo` |
| `--language` | Idioma del audio (si se conoce) | `Spanish`, `French`, `English`, etc. o código ISO (`es`, `fr`, `en`) |
| `--task` | Qué hacer con el audio | `transcribe` (por defecto) o `translate` (traduce al inglés) |
| `--output_format` | Formato de salida | `txt`, `srt`, `vtt`, `tsv`, `json`, `all` |
| `--output_dir` | Carpeta de salida | Ruta (ej. `/content/resultados`) |
| `--word_timestamps` | Marcas de tiempo por palabra | `True` o `False` |
| `--initial_prompt` | Contexto/vocabulario para guiar la transcripción | Texto libre |
| `--condition_on_previous_text` | Usar texto previo como contexto | `True` (defecto) o `False` |
| `--verbose` | Mostrar progreso en pantalla | `True` o `False` |

### Sobre `--initial_prompt`
Este parámetro es **muy útil** para mejorar la transcripción. Puedes usarlo para:
- Dar **nombres propios** o **términos técnicos** que Whisper podría no reconocer.
- Indicar el **estilo de puntuación** deseado.
- Ejemplo: `--initial_prompt "Entrevista con el Dr. Pérez sobre los arrecifes de coral en Bocas del Toro."`

### Sobre `--condition_on_previous_text`
- En `True` (por defecto): Whisper usa la transcripción anterior como contexto. Esto mejora la coherencia pero puede causar **"alucinaciones"** (texto repetido o inventado) al final de audios largos.
- En `False`: cada segmento se transcribe de forma independiente. Más seguro para audios muy largos.


### 📝 Ejemplo 1: Transcripción básica (calidad máxima)

In [ ]:
# Crear carpeta para los resultados
!mkdir -p /content/resultados

# Transcripción con el modelo large (máxima calidad)
# Genera TODOS los formatos de salida (txt, srt, vtt, tsv, json)
!whisper "{AUDIO_FILE}" \
    --model large \
    --language Spanish \
    --task transcribe \
    --output_format all \
    --output_dir /content/resultados \
    --verbose True

### 📝 Ejemplo 2: Transcripción rápida (modelo turbo)

In [ ]:
# Transcripción rápida con turbo (ideal para audios largos)
!whisper "{AUDIO_FILE}" \
    --model turbo \
    --language Spanish \
    --output_format all \
    --output_dir /content/resultados \
    --verbose True

### 📝 Ejemplo 3: Transcripción con vocabulario específico y marcas de tiempo por palabra

In [ ]:
# Transcripción con initial_prompt para guiar nombres propios / vocabulario
# y word_timestamps para obtener el tiempo de cada palabra
!whisper "{AUDIO_FILE}" \
    --model large \
    --language Spanish \
    --output_format all \
    --output_dir /content/resultados \
    --word_timestamps True \
    --initial_prompt "Entrevista sobre patrimonio cultural con María Fernanda López y el arquitecto Juan Carlos Restrepo." \
    --verbose True

### 📝 Ejemplo 4: Traducir un audio al inglés

In [ ]:
# Traducción al inglés (solo funciona con medium o large, NO con turbo)
!whisper "{AUDIO_FILE}" \
    --model large \
    --language Spanish \
    --task translate \
    --output_format all \
    --output_dir /content/resultados \
    --verbose True

### 📝 Ejemplo 5: Audios muy largos o con "alucinaciones"

Si Whisper empieza a repetir texto o a inventar palabras al final de un audio largo, desactiva `condition_on_previous_text`:

In [ ]:
# Para audios muy largos o con problemas de repetición
!whisper "{AUDIO_FILE}" \
    --model large \
    --language Spanish \
    --output_format all \
    --output_dir /content/resultados \
    --condition_on_previous_text False \
    --verbose True

---
## Método B: Script Python (más control)

Este método permite manipular directamente el resultado de la transcripción en Python. Útil si quieres procesar la transcripción después (por ejemplo, limpiar texto, contar palabras, etc.).

In [ ]:
import whisper
import torch

# ===== CONFIGURACIÓN (edita según tus necesidades) =====
MODELO = "large"          # tiny | base | small | medium | large | turbo
IDIOMA = "es"             # es, fr, en, pt, de, it, ja, zh, etc. o None para autodetección
TAREA = "transcribe"      # transcribe | translate (traducir al inglés)
PROMPT = None             # Texto de contexto, ej: "Reunión con el profesor García sobre botánica."
TIMESTAMPS_PALABRA = False # True para obtener tiempos por palabra
# ========================================================

print(f"⏳ Cargando modelo '{MODELO}'...")
model = whisper.load_model(MODELO)
print(f"✅ Modelo cargado en {'GPU' if next(model.parameters()).is_cuda else 'CPU'}")

print(f"\n⏳ Transcribiendo: {AUDIO_FILE}")
result = model.transcribe(
    AUDIO_FILE,
    language=IDIOMA,
    task=TAREA,
    initial_prompt=PROMPT,
    word_timestamps=TIMESTAMPS_PALABRA,
    verbose=True
)

print("\n" + "="*60)
print("📝 TRANSCRIPCIÓN COMPLETA:")
print("="*60)
print(result["text"])

### Guardar el resultado en diferentes formatos

El resultado del script Python se puede exportar fácilmente a `.txt`, `.srt`, `.vtt`, etc., usando las utilidades integradas de Whisper:

In [ ]:
from whisper.utils import get_writer
import os

# Carpeta de salida
OUTPUT_DIR = "/content/resultados"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Nombre base del archivo (sin extensión)
audio_basename = os.path.splitext(os.path.basename(AUDIO_FILE))[0]

# Guardar en todos los formatos disponibles
for fmt in ["txt", "srt", "vtt", "tsv", "json"]:
    writer = get_writer(fmt, OUTPUT_DIR)
    writer(result, audio_basename)
    print(f"✅ Guardado: {OUTPUT_DIR}/{audio_basename}.{fmt}")

print(f"\n📂 Todos los archivos guardados en: {OUTPUT_DIR}")

---
# 📄 8. Explorar y descargar los resultados

### Formatos de salida

| Formato | Extensión | Descripción | Uso típico |
|---------|-----------|-------------|------------|
| Texto plano | `.txt` | Solo el texto, sin marcas de tiempo | Lectura, búsqueda, análisis de texto |
| SubRip | `.srt` | Subtítulos con marcas de tiempo | Videos, VLC, YouTube, editores de video |
| WebVTT | `.vtt` | Subtítulos web con marcas de tiempo | Navegadores, HTML5, plataformas web |
| TSV | `.tsv` | Datos tabulados (inicio, fin, texto) | Hojas de cálculo, análisis |
| JSON | `.json` | Datos completos y estructurados | Programación, procesamiento posterior |

In [ ]:
# Listar los archivos generados
import os

OUTPUT_DIR = "/content/resultados"
print(f"📂 Archivos en {OUTPUT_DIR}:\n")
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(filepath) / 1024
    print(f"  📄 {f:40s} ({size:.1f} KB)")

### Leer la transcripción en texto plano

In [ ]:
# Leer y mostrar el archivo .txt
import glob

txt_files = glob.glob("/content/resultados/*.txt")
if txt_files:
    with open(txt_files[0], 'r', encoding='utf-8') as f:
        text = f.read()
    print(f"📄 Archivo: {txt_files[0]}\n")
    print(text[:3000])  # Primeros 3000 caracteres
    if len(text) > 3000:
        print(f"\n... [truncado, total: {len(text)} caracteres]")
else:
    print("No se encontraron archivos .txt en /content/resultados/")

### Leer los subtítulos (SRT)

In [ ]:
# Leer y mostrar el archivo .srt (primeras 20 entradas)
import glob

srt_files = glob.glob("/content/resultados/*.srt")
if srt_files:
    with open(srt_files[0], 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"📄 Archivo: {srt_files[0]}\n")
    # Mostrar las primeras 60 líneas (aprox. 20 subtítulos)
    print(''.join(lines[:60]))
    if len(lines) > 60:
        print(f"\n... [truncado, total: {len(lines)} líneas]")
else:
    print("No se encontraron archivos .srt en /content/resultados/")

### 💾 Descargar los resultados

In [ ]:
from google.colab import files
import os

OUTPUT_DIR = "/content/resultados"

# Descargar todos los archivos generados
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    print(f"📥 Descargando: {f}")
    files.download(filepath)

### O copiar los resultados a Google Drive

In [ ]:
# Copiar resultados a Google Drive
import shutil

# ✏️ EDITA la carpeta destino en Drive
DRIVE_DEST = "/content/drive/MyDrive/Transcripciones_Whisper"

os.makedirs(DRIVE_DEST, exist_ok=True)

for f in os.listdir("/content/resultados"):
    src = os.path.join("/content/resultados", f)
    dst = os.path.join(DRIVE_DEST, f)
    shutil.copy2(src, dst)
    print(f"✅ Copiado: {f} → {DRIVE_DEST}/")

print(f"\n📂 Todos los archivos copiados a: {DRIVE_DEST}")

---
# 💡 9. Consejos y resolución de problemas

### Audio de mala calidad
- Usa el modelo `large` (más robusto con ruido de fondo, acentos, etc.).
- Especifica el `--language` en lugar de dejar la detección automática.
- Usa `--initial_prompt` para dar contexto y vocabulario técnico.

### Whisper repite texto o "alucina"
- Usa `--condition_on_previous_text False`.
- Baja la temperatura: en el script Python, `temperature=0.0` produce resultados más estables.

### El audio es muy largo (> 1 hora)
- `turbo` es mucho más rápido que `large` (~8× más rápido).
- Si la sesión de Colab se desconecta, recuerda que las sesiones gratuitas tienen un **límite de ~12 horas** y pueden cortarse por inactividad.
- Consejo: guarda resultados parciales en Google Drive.

### Error de memoria (CUDA out of memory)
- Usa un modelo más pequeño (`medium` o `small`).
- Reinicia el entorno de ejecución (menú *Entorno de ejecución → Reiniciar entorno de ejecución*) para liberar la GPU.

### Idiomas soportados
Whisper soporta más de **96 idiomas**. Algunos de los más relevantes:

| Código | Idioma | Código | Idioma |
|--------|--------|--------|--------|
| `es` | Español | `en` | Inglés |
| `fr` | Francés | `pt` | Portugués |
| `de` | Alemán | `it` | Italiano |
| `ja` | Japonés | `zh` | Chino |
| `ko` | Coreano | `ar` | Árabe |
| `nl` | Neerlandés | `ru` | Ruso |
| `ca` | Catalán | `eu` | Euskera |

Para la lista completa: https://github.com/openai/whisper/blob/main/whisper/tokenizer.py

---
# 🚀 Referencia rápida (*para ejecutar fácilmente en otros entornos*)

```bash
# Transcripción básica con máxima calidad
!whisper "mi_audio.mp3" --model large --language Spanish --output_format all --output_dir /content/resultados

# Transcripción rápida
!whisper "mi_audio.mp3" --model turbo --language Spanish --output_format all --output_dir /content/resultados

# Con vocabulario específico
!whisper "mi_audio.mp3" --model large --language Spanish --initial_prompt "Reunión con el Dr. Pérez" --output_format all --output_dir /content/resultados

# Traducir al inglés
!whisper "mi_audio.mp3" --model large --language Spanish --task translate --output_format all --output_dir /content/resultados

# Descargar audio de YouTube primero
!yt-dlp --extract-audio --audio-format mp3 --audio-quality 0 -o "/content/audio_yt.%(ext)s" "URL_DEL_VIDEO"
```

---
*Cuaderno creado redactada por **el grupo de interés en adquisición asistida de textos** del **consorcio ARIANE** — Marzo 2026*

*Basado en [OpenAI Whisper](https://github.com/openai/whisper) (licencia MIT)*